In [4]:
import os
import pandas as pd

uxm_results_path = "/data/dmytro/cfSortData/UXM_results"
import numpy as np

In [5]:
metadata = pd.read_csv(
    "/data/dmytro/cfSortData/GSE233417_samples/GSE233417_samples.csv"
)

In [6]:
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

calibrator = LinearCalibrator()
calibrator.load_calibration_parameters("/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/pseudobulk/uxm_results_pseudobulk/uxm_linear_calibrator.npz")

In [7]:
cfsort_tissue_to_celltypes = {
    "adipose tissue": ["Adipocytes"],
    "vagina": ["Epid-Kerat", "Smooth-Musc"],
    "skin": ["Epid-Kerat", "Dermal-Fibro"],
    "skin exposed": ["Epid-Kerat", "Dermal-Fibro"],
    "small intestine": ["Small-Int-Ep"],
    "esophagus": ["Epid-Kerat"],
    "esophagus gast junc": ["Gastric-Ep", "Epid-Kerat"],
    "esophagus muscularis": ["Smooth-Musc"],
    "esophagus musc": ["Smooth-Musc"],
    "kidney": ["Kidney-Ep"],
    "salivary gland": ["Head-Neck-Ep"],
    "prostate": ["Prostate-Ep"],
    "breast": ["Breast-Luminal-Ep", "Breast-Basal-Ep"],
    "nerve": ["Neuron", "Oligodend"],
    "pituitary": ["Neuron"],
    "pancreas": [
        "Pancreas-Acinar",
        "Pancreas-Beta",
        "Pancreas-Alpha",
        "Pancreas-Delta",
        "Pancreas-Duct",
    ],
    "testis": ["Epid-Kerat"],  # Sertoli / seminiferous epithelium → closest proxy
    "muscle": ["Skeletal-Musc"],
    "adrenal gland": ["Endothel"],  # no perfect match; endothelial + stromal dominant
    "blood vessel": ["Endothel", "Smooth-Musc"],
    "blood vessel coronary": ["Endothel", "Smooth-Musc"],
    "heart": ["Heart-Cardio", "Heart-Fibro"],
    "heart atrial": ["Heart-Cardio"],
    "spleen": ["Blood-B", "Blood-T", "Blood-Mono+Macro"],
    "ovary": ["Ovary-Ep"],
    "bladder": ["Bladder-Ep"],
    "cervix uteri": ["Epid-Kerat"],
    "cervix uteri endocervix": ["Head-Neck-Ep"],  # columnar glandular epithelium
    "uterus": ["Smooth-Musc"],  # myometrium dominates by mass
    "fallopian tube": ["Fallopian-Ep"],
    "thyroid": ["Thyroid-Ep"],
    "colon": ["Colon-Ep", "Colon-Fibro"],
    "liver": ["Liver-Hep"],
    "stom": ["Gastric-Ep"],  # stomach
    "lung": ["Lung-Ep-Alveo", "Lung-Ep-Bron"],
    "WBC": ["Blood-T", "Blood-B", "Blood-Mono+Macro", "Blood-NK", "Blood-Granul"],
}

In [35]:
results_top25 = []
results_top25gsc = []
results_top250 = []
for file in os.listdir(uxm_results_path):
    sub = pd.read_csv(os.path.join(uxm_results_path, file))
    if "Atlas.U250" in file:
        sub.columns = ["CellType", "UXM_U250"]
    elif "U25gscSorted" in file:
        sub.columns = ["CellType", "UXM_U25_GSC_Sorted"]
    else:
        sub.columns = ["CellType", "UXM_U25"]
        sub["UXM_U25_LCal"] = calibrator.predict(
            np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39))
        )[0][0]
    sub["file"] = file.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub["Biosample term id"] = "not mapped"
    sub_meta = metadata[metadata["sample_geo_accession"] == file.split("_")[0]]
    if len(sub_meta) != 1:
        raise ValueError(f"Corrupted metadata for {file}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    sub["AuditGood"] = True
    sub["CellTypeProxy"] = [cfsort_tissue_to_celltypes[tissue]] * 39
    sub["TagFiltered"] = True
    if "Atlas.U250" in file:
        results_top250.append(sub)
    elif "U25gscSorted" in file:
        results_top25gsc.append(sub)
    else:
        results_top25.append(sub)

In [36]:
results_top25_pd = pd.concat(results_top25, axis=0).reset_index()
results_top250_pd = pd.concat(results_top250, axis=0).reset_index()
results_top25gsc = pd.concat(results_top25gsc, axis=0).reset_index()

In [38]:
results_uxm_all = pd.merge(pd.merge(results_top25_pd, results_top250_pd[["CellType", "file", "UXM_U250"]], on=["CellType", "file"]), results_top25gsc[["CellType", "file", "UXM_U25_GSC_Sorted"]], on =["CellType", "file"])

In [39]:
results_uxm_all.drop("index", axis=1, inplace=True)

In [40]:
results_uxm_all

,CellType,UXM_U25,UXM_U25_LCal,file,Biosample organism,Biosample type,Biosample term id,Biosample term name,AuditGood,CellTypeProxy,TagFiltered,UXM_U250,UXM_U25_GSC_Sorted
0,Adipocytes,0.186934,0.212856,GSM7427522,Homo sapiens,tissue,not mapped,colon,True,"[Colon-Ep, Colon-Fibro]",True,0.017605,0.084847
1,Bladder-Ep,0.000000,0.000000,GSM7427522,Homo sapiens,tissue,not mapped,colon,True,"[Colon-Ep, Colon-Fibro]",True,0.000000,0.000000
2,Blood-B,0.004218,0.001176,GSM7427522,Homo sapiens,tissue,not mapped,colon,True,"[Colon-Ep, Colon-Fibro]",True,0.000000,0.010685
3,Blood-Granul,0.000000,0.000000,GSM7427522,Homo sapiens,tissue,not mapped,colon,True,"[Colon-Ep, Colon-Fibro]",True,0.000000,0.000000
4,Blood-Mono+Macro,0.010355,0.007129,GSM7427522,Homo sapiens,tissue,not mapped,colon,True,"[Colon-Ep, Colon-Fibro]",True,0.138522,0.101797
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20314,Prostate-Ep,0.000000,0.000000,GSM7427538,Homo sapiens,tissue,not mapped,stom,True,[Gastric-Ep],True,0.002102,0.000000
20315,Skeletal-Musc,0.000000,0.000000,GSM7427538,Homo sapiens,tissue,not mapped,stom,True,[Gastric-Ep],True,0.012034,0.000000
20316,Small-Int-Ep,0.015862,0.015302,GSM7427538,Homo sapiens,tissue,not mapped,stom,True,[Gastric-Ep],True,0.007465,0.012604
20317,Smooth-Musc,0.000000,0.000000,GSM7427538,Homo sapiens,tissue,not mapped,stom,True,[Gastric-Ep],True,0.058405,0.065058


In [32]:
results_uxm_all.to_csv("cfSort_rrbs_uxm_results.csv", index=False)

### Load other results

In [1]:
#target_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_soft_labels_pooled_jakkard/pseudobulk/deconvolutions"
# target_path = "/data/dmytro/methyldl_models/archive/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels/deconvolutions"
target_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/deconvolutions"


In [2]:
file_to_model_dict = {
 'deconvolution_xgboost.csv':"XGB",
 'deconvolution_xgboost_callibrated.csv':"XGB_LCal",
 'deconvolution_3Layer_MLP.csv': "MLP",
 'deconvolution_3Layer_MLP_callibrated.csv':"MLP_LCal",
 'deconvolution_Shallow_Wide_Network.csv': "SWN",
 'deconvolution_Shallow_Wide_Network_callibrated.csv':"SWN_LCal",
 'deconvolution_nnls.csv':"NNLS",
 'deconvolution_nnls_callibrated.csv':"NNLS_LCal",
 'deconvolution_psls.csv': "PSLS",
 'deconvolution_psls_callibrated.csv':"PSLS_LCal",
}

In [8]:
results_methylbert_soft = []
for folder in os.listdir(target_path):
    sub = pd.DataFrame()
    for key,value in file_to_model_dict.items():
        partial_sub = pd.read_csv(os.path.join(target_path,folder,key))
        if not len(sub):
            sub = partial_sub
            sub.columns = ["CellType", value]
        else:
            sub[value] = partial_sub[partial_sub.columns[1]]
    sub["file"] = folder.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub["Biosample term id"] = "not mapped"  
    sub_meta = metadata[metadata["sample_geo_accession"] == folder.split("_")[0]]
    if len(sub_meta) !=1:
        raise ValueError(f"Corrupted metadata for {folder}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    sub["AuditGood"] = True
    sub["CellTypeProxy"] = [cfsort_tissue_to_celltypes[tissue]]*39
    sub["TagFiltered"] = True
    results_methylbert_soft.append(sub)      

In [9]:
results_methylbert_soft = pd.concat(results_methylbert_soft, axis=0).reset_index()

In [10]:
results_methylbert_soft["classifier"] = "MethylBERT"
results_methylbert_soft["labeling_scheme"] = "Hard Labels"

In [11]:
results_methylbert_soft.to_csv("cfSort_rrbs_methylbert_hard_results.csv", index=False)